# Kimi-VL-A3B-Thinking-2506 4bit を Google Colab で動かす


In [ ]:
# =========================================
# コード1 初回だけ実行：Pillowを完全修復 + 依存関係を固定
# 実行後にランタイムが自動再起動します
# =========================================
import os
import sys
import site
import glob
import shutil
import subprocess

print("Python:", sys.version)

# GPU確認
subprocess.run(["nvidia-smi", "-L"], check=False)

# -------------------------------------------------
# 1) Pillowをpip上からアンインストール
# -------------------------------------------------
subprocess.run(
    [sys.executable, "-m", "pip", "uninstall", "-y", "Pillow", "pillow"],
    check=False,
)

# -------------------------------------------------
# 2) pip uninstallで残ることがあるPIL本体・dist-infoを物理削除
#    Python 3.12などパスが変わってもsite-packagesを自動取得
# -------------------------------------------------
candidate_dirs = []

for p in site.getsitepackages():
    candidate_dirs.append(p)

user_site = site.getusersitepackages()
if user_site:
    candidate_dirs.append(user_site)

for base in dict.fromkeys(candidate_dirs):
    if not os.path.isdir(base):
        continue

    targets = [
        os.path.join(base, "PIL"),
        *glob.glob(os.path.join(base, "Pillow-*.dist-info")),
        *glob.glob(os.path.join(base, "pillow-*.dist-info")),
    ]

    for target in targets:
        if os.path.isdir(target):
            print("remove:", target)
            shutil.rmtree(target, ignore_errors=True)
        elif os.path.isfile(target):
            print("remove:", target)
            try:
                os.remove(target)
            except OSError:
                pass

# -------------------------------------------------
# 3) Kimi-VL側の推奨に近いTransformers環境を構築
#    Pillow 11.3.0 は Python 3.12 対応
# -------------------------------------------------
packages = [
    "Pillow==11.3.0",
    "transformers==4.48.2",
    "accelerate>=1.2,<2",
    "bitsandbytes>=0.45.0",
    "sentencepiece",
    "safetensors",
    "huggingface_hub",
]

subprocess.run(
    [
        sys.executable, "-m", "pip", "install",
        "--no-cache-dir",
        "--upgrade",
        *packages,
    ],
    check=True,
)

print()
print("==============================================")
print("依存関係のインストールが完了しました。")
print("Pillowの古いモジュールを確実に捨てるため再起動します。")
print("再接続後はコード2から実行してください。")
print("==============================================")

# Colabランタイムを強制再起動
os.kill(os.getpid(), 9)


Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]


## 再起動後はここから

コード1の実行後、Colabが一度切断されます。再接続されたら、**コード2から下だけを実行してください。**

最初にPillowとTransformersが正しく読み込めることを検査します。


In [2]:
# =========================================
# コード2 環境チェック
# =========================================

import sys
import torch
import PIL
import transformers
import bitsandbytes as bnb

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Pillow:", PIL.__version__)
print("bitsandbytes:", bnb.__version__)

# Pillowの基本機能だけ確認
from PIL import Image, ImageDraw, ImageFont

print("PIL.Image: OK")
print("PIL.ImageDraw: OK")
print("PIL.ImageFont: OK")

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPUが有効ではありません。Colabの「ランタイム」→"
        "「ランタイムのタイプを変更」でGPUを選択してください。"
    )

print("GPU:", torch.cuda.get_device_name(0))
props = torch.cuda.get_device_properties(0)
print("GPU memory: %.1f GB" % (props.total_memory / 1024**3))


Python: 3.12.13
PyTorch: 2.11.0+cu128
Transformers: 4.48.2
Pillow: 11.3.0
bitsandbytes: 0.50.0
PIL.Image: OK
PIL.ImageDraw: OK
PIL.ImageFont: OK
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
GPU memory: 95.0 GB


In [3]:
# =========================================
# コード3 Google Driveとキャッシュ設定
# =========================================
from google.colab import drive
from pathlib import Path
import os
import shutil

drive.mount("/content/drive")

PROJECT_DIR = Path("/content/drive/MyDrive/Colab Notebooks/LocalLLM")
USE_DRIVE_CACHE = True

if USE_DRIVE_CACHE:
    CACHE_DIR = PROJECT_DIR / "Program" / "hf_cache_kimi"
else:
    CACHE_DIR = Path("/content/hf_cache_kimi")

CACHE_DIR.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(CACHE_DIR)
os.environ["HF_HUB_CACHE"] = str(CACHE_DIR)

usage = shutil.disk_usage(CACHE_DIR)
free_gb = usage.free / 1024**3

print("CACHE_DIR:", CACHE_DIR)
print("free space: %.1f GB" % free_gb)

if free_gb < 20:
    print("WARNING: モデルとキャッシュのため20GB以上の空きを推奨します。")


Mounted at /content/drive
CACHE_DIR: /content/drive/MyDrive/Colab Notebooks/LocalLLM/Program/hf_cache_kimi
free space: 179.1 GB


In [4]:
# =========================================
# コード4 Kimi-VL 4bitモデルとProcessorを読み込む
# =========================================
import torch
from transformers import AutoModelForCausalLM, AutoProcessor

MODEL_ID = "SoybeanMilk/Kimi-VL-A3B-Thinking-2506-BNB-4bit"

print("Loading:", MODEL_ID)
print("初回はモデルのダウンロードに時間がかかります。")

# 量子化情報はモデルのconfigに保存済みなので、
# BitsAndBytesConfigを二重指定しない。
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    device_map="auto",
    cache_dir=str(CACHE_DIR),
    low_cpu_mem_usage=True,
)

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    cache_dir=str(CACHE_DIR),
)

model.eval()

# device_map="auto" では model.device が必ずしも入力先とは限らないため、
# embeddingのdeviceを優先して取得する。
try:
    INPUT_DEVICE = model.get_input_embeddings().weight.device
except Exception:
    INPUT_DEVICE = model.device

print("model loaded:", MODEL_ID)
print("input device:", INPUT_DEVICE)
print("4bit:", getattr(model, "is_loaded_in_4bit", False))
print("GPU allocated: %.2f GB" % (torch.cuda.memory_allocated() / 1024**3))
print("GPU reserved : %.2f GB" % (torch.cuda.memory_reserved() / 1024**3))


Loading: SoybeanMilk/Kimi-VL-A3B-Thinking-2506-BNB-4bit
初回はモデルのダウンロードに時間がかかります。


config.json: 0.00B [00:00, ?B/s]

configuration_kimi_vl.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/moonshotai/Kimi-VL-A3B-Thinking-2506:
- configuration_kimi_vl.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


modeling_kimi_vl.py: 0.00B [00:00, ?B/s]

...

image_processing_kimi_vl.py: 0.00B [00:00, ?B/s]

tokenization_moonshot.py: 0.00B [00:00, ?B/s]

tiktoken.model:   0%|          | 0.00/2.80M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

model loaded: SoybeanMilk/Kimi-VL-A3B-Thinking-2506-BNB-4bit
input device: cuda:0
4bit: True
GPU allocated: 9.94 GB
GPU reserved : 9.97 GB


In [5]:
# =========================================
# コード5 応答生成関数
# テキストのみ / 画像付きの両方に対応
# =========================================
from PIL import Image
import torch

def _extract_summary(text: str):
    """
    Kimi-VL Thinkingが思考タグを返した場合、最終回答部分を優先して返す。
    タグがない場合は全文を返す。
    """
    bot = "◁think▷"
    eot = "◁/think▷"

    if eot in text:
        return text.split(eot, 1)[1].strip()

    return text.strip()


@torch.inference_mode()
def kimi_generate(
    user_text,
    image=None,
    max_new_tokens=512,
    temperature=0.7,
    show_thinking=False,
):
    user_text = (user_text or "").strip()

    if not user_text:
        return ""

    content = []

    if image is not None:
        content.append({"type": "image", "image": ""})

    content.append({"type": "text", "text": user_text})

    messages = [
        {
            "role": "user",
            "content": content,
        }
    ]

    # Kimi-VLのchat templateを適用
    text = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
    )

    if image is not None:
        if not isinstance(image, Image.Image):
            image = Image.fromarray(image)
        image = image.convert("RGB")

        inputs = processor(
            images=[image],
            text=text,
            return_tensors="pt",
            padding=True,
            truncation=True,
        )
    else:
        inputs = processor(
            text=text,
            return_tensors="pt",
            padding=True,
            truncation=True,
        )

    inputs = {
        k: v.to(INPUT_DEVICE) if hasattr(v, "to") else v
        for k, v in inputs.items()
    }

    generation_kwargs = {
        **inputs,
        "max_new_tokens": int(max_new_tokens),
        "do_sample": True,
        "temperature": float(temperature),
        "use_cache": True,
    }

    # pad/eosが定義されていれば利用
    tokenizer = getattr(processor, "tokenizer", None)
    if tokenizer is not None:
        if tokenizer.eos_token_id is not None:
            generation_kwargs["eos_token_id"] = tokenizer.eos_token_id
        if tokenizer.pad_token_id is not None:
            generation_kwargs["pad_token_id"] = tokenizer.pad_token_id
        elif tokenizer.eos_token_id is not None:
            generation_kwargs["pad_token_id"] = tokenizer.eos_token_id

    generated_ids = model.generate(**generation_kwargs)

    input_len = inputs["input_ids"].shape[-1]
    generated_ids_trimmed = generated_ids[:, input_len:]

    response = processor.batch_decode(
        generated_ids_trimmed,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0].strip()

    if show_thinking:
        return response

    return _extract_summary(response)


In [6]:
# =========================================
# コード6 テキストチャット動作確認
# =========================================
reply = kimi_generate(
    "日本語で1文だけ自己紹介してください。",
    image=None,
    max_new_tokens=256,
    temperature=0.7,
    show_thinking=False,
)

print(reply)


◁think▷まず、ユーザーからの依頼内容を分析します。ユーザーは日本語で1文だけ自己紹介を求めています。これは非常に簡素な依頼であり、自分の自己紹介を適切にまとめ、相手に分かりやすく伝えることが求められていると思います。

次に、自分の自己紹介を考えます。基本的には、名前、生年月日、職業、興味、そして最後に「よろしくお願いします」のような構成が一般的です。日本の自己紹介の习惯とも考え合い、簡潔さを重視します。

日本語での表現を検討します。日本語で書く際には、丁寧さと簡潔さをバランスよく取ることが重要です。例えば、「私は」という表現を使うか、直接的な「私は…」という表現を選ぶか、両方の可能性があります。

そして、最後に「よろしくお願いします」という表


## 任意：画像付き推論

以下のセルは、手元の画像をアップロードしてKimi-VLに質問する簡単な確認用です。


In [7]:
# =========================================
# コード7 任意：画像付き推論
# =========================================
from google.colab import files
from PIL import Image
import io

uploaded = files.upload()

if uploaded:
    filename = next(iter(uploaded))
    test_image = Image.open(io.BytesIO(uploaded[filename])).convert("RGB")
    display(test_image)

    print(
        kimi_generate(
            "この画像について日本語で簡潔に説明してください。",
            image=test_image,
            max_new_tokens=512,
            temperature=0.7,
            show_thinking=False,
        )
    )


Output hidden; open in https://colab.research.google.com to view.

In [9]:
# =========================================
# コード8 Gradioを用いたKimi-VLチャットUI
# =========================================
import gradio as gr

print("Gradio:", gr.__version__)

def gr_chat(user_text, image):
    user_text = (user_text or "").strip()

    if not user_text:
        return "質問を入力してください。"

    try:
        reply = kimi_generate(
            user_text=user_text,
            image=image,
            max_new_tokens=512,
            temperature=0.7,
            show_thinking=False,
        )
        return reply

    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        return "CUDA out of memory が発生しました。画像を外すか、ランタイムを再起動してください。"

    except Exception as e:
        return f"{type(e).__name__}: {e}"


with gr.Blocks(title="Kimi-VL Chat") as demo:
    gr.Markdown("## Kimi-VL Chat")

    image_box = gr.Image(
        label="画像（任意）",
        type="pil",
    )

    user_box = gr.Textbox(
        label="質問",
        placeholder="質問を入力してください。",
        lines=3,
    )

    with gr.Row():
        send_btn = gr.Button("Send", variant="primary")
        clear_btn = gr.Button("Clear")

    output_box = gr.Textbox(
        label="回答",
        lines=12,
    )

    send_btn.click(
        fn=gr_chat,
        inputs=[user_box, image_box],
        outputs=output_box,
        queue=False,
    )

    # Enterキーでも送信
    user_box.submit(
        fn=gr_chat,
        inputs=[user_box, image_box],
        outputs=output_box,
        queue=False,
    )

    clear_btn.click(
        lambda: ("", None, ""),
        outputs=[user_box, image_box, output_box],
        queue=False,
    )

print("WARNING: share=True で一時的な公開URLを作成します。")

demo.launch(
    share=True,
    inline=True,
    debug=False,
)

Gradio: 6.20.0
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://0043bafb15fc475415.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
